# 02 - Estandarización de Schema

**Objetivo**: Estandarizar las columnas de todas las facturas a un schema único.

**Input**: `data/staging/{mes}_raw.csv`

**Output**: `data/staging/{mes}_standardized.csv`

**Responsabilidades**:
- Mapear diferentes esquemas de columnas a uno estándar
- Calcular columnas faltantes (ej: Total a partir de Cantidad × Precio)
- Asegurar que todas las filas tengan el mismo schema

**Schema objetivo**:
```
- Producto: str
- Cantidad: float
- Unidad: str
- Valor_Unitario: float
- Total: float
- Tienda: str
- Fecha: str (YYYY-MM-DD)
- Mes: int
- Año: int
- Categoria: str
```

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Configuración

In [2]:
# Configurar el mes a procesar
MES = "enero"
ANIO = 2026

# Rutas
STAGING_PATH = Path("../data/staging")
INPUT_FILE = STAGING_PATH / f"{MES}_{ANIO}_raw.csv"
OUTPUT_FILE = STAGING_PATH / f"{MES}_{ANIO}_standardized.csv"

print(f"📂 Archivo de entrada: {INPUT_FILE}")
print(f"💾 Archivo de salida: {OUTPUT_FILE}")

📂 Archivo de entrada: ../data/staging/enero_2026_raw.csv
💾 Archivo de salida: ../data/staging/enero_2026_standardized.csv


## 1. Cargar datos de staging

In [3]:
# Cargar datos
df = pd.read_csv(INPUT_FILE)

print(f"✅ Datos cargados:")
print(f"   - Filas: {len(df)}")
print(f"   - Columnas: {list(df.columns)}")

df.head()

✅ Datos cargados:
   - Filas: 94
   - Columnas: ['Producto', 'Cantidad', 'Unidad', 'Precio', 'tienda', 'fecha', 'mes', 'año', 'Item', 'Descripcion', 'Valor_Unitario', 'Total', 'Ref']


,Producto,Cantidad,Unidad,Precio,tienda,fecha,mes,año,Item,Descripcion,Valor_Unitario,Total,Ref
0,Yogurt griego,1.0,und,7300.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN
1,Jamón Pietrán,1.0,und,13650.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN
2,Queso parmesano,1.0,und,14850.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN
3,Quesillo tajado,1.0,und,9990.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN
4,Topping mediano,4.0,und,19800.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN


## 2. Analizar esquemas existentes

In [4]:
# Identificar qué columnas tiene cada tienda
print("🔍 Esquemas por tienda:\n")

for tienda in df['tienda'].unique():
    df_tienda = df[df['tienda'] == tienda]
    # Excluir columnas de metadata para ver solo las de datos
    cols_datos = [col for col in df_tienda.columns if col not in ['tienda', 'fecha', 'mes', 'año']]
    print(f"  {tienda}:")
    print(f"    Columnas: {cols_datos}")
    print()

🔍 Esquemas por tienda:

  d1:
    Columnas: ['Producto', 'Cantidad', 'Unidad', 'Precio', 'Item', 'Descripcion', 'Valor_Unitario', 'Total', 'Ref']

  surtiplaza:
    Columnas: ['Producto', 'Cantidad', 'Unidad', 'Precio', 'Item', 'Descripcion', 'Valor_Unitario', 'Total', 'Ref']

  mercacentro:
    Columnas: ['Producto', 'Cantidad', 'Unidad', 'Precio', 'Item', 'Descripcion', 'Valor_Unitario', 'Total', 'Ref']



## 3. Estandarizar columnas

Tenemos 2 esquemas principales:

**Esquema A** (algunas tiendas):
- Producto, Cantidad, Unidad, Precio

**Esquema B** (otras tiendas):
- Item/Ref, Descripcion, Cantidad, Unidad, Valor_Unitario, Total

**Schema objetivo**:
- Producto, Cantidad, Unidad, Valor_Unitario, Total

In [8]:
def estandarizar(df):
    """
    Estandariza al schema objetivo.
    """

    df = df.copy()

    # 1. PRODUCTO
    df["Producto"] = df["Producto"].fillna(df["Descripcion"])

    df["Producto"] = df["Producto"].astype(str).str.lower()

    # 2. TOTAL
    # Si Total está vacío, usar Valor_Unitario o el precio
    df["Total"] = df["Total"].fillna(df["Precio"])

    # 3. VALOR_UNITARIO
    # Calcular cuando no exista
    df["Valor_Unitario"] = df["Total"] / df["Cantidad"]

    # Seleccionar solo columnas finales
    df_final = df[
        [
            "Producto",
            "Cantidad",
            "Unidad",
            "Total",
            "Valor_Unitario",
            "tienda",
            "fecha",
            "mes",
            "año",
        ]
    ]

    return df_final


# Aplicar estandarización
print("⚙️ Estandarizando filas...")
df_standardized = estandarizar(df)

print(f"✅ Estandarización completa")
print(f"   - Columnas finales: {list(df_standardized.columns)}")

⚙️ Estandarizando filas...
✅ Estandarización completa
   - Columnas finales: ['Producto', 'Cantidad', 'Unidad', 'Total', 'Valor_Unitario', 'tienda', 'fecha', 'mes', 'año']


In [9]:
df_standardized.head()

,Producto,Cantidad,Unidad,Total,Valor_Unitario,tienda,fecha,mes,año
0,yogurt griego,1.0,und,7300.0,7300.0,d1,2026-01-11,1,2026
1,jamón pietrán,1.0,und,13650.0,13650.0,d1,2026-01-11,1,2026
2,queso parmesano,1.0,und,14850.0,14850.0,d1,2026-01-11,1,2026
3,quesillo tajado,1.0,und,9990.0,9990.0,d1,2026-01-11,1,2026
4,topping mediano,4.0,und,19800.0,4950.0,d1,2026-01-11,1,2026


## 4. Validar estandarización

In [11]:
# Verificar que todas las filas tienen las columnas esperadas
expected_cols = ['Producto', 'Cantidad', 'Unidad', 'Valor_Unitario', 'Total', 
                 'tienda', 'fecha', 'mes', 'año']

print("🔍 Validación de columnas:")
for col in expected_cols:
    if col in df_standardized.columns:
        print(f"   ✅ {col}")
    else:
        print(f"   ❌ {col} - FALTANTE")

# Verificar valores nulos
print("\n🔍 Valores nulos por columna:")
print(df_standardized.isnull().sum())

# Vista previa
print("\n📋 Vista previa del dataset estandarizado:")
df_standardized.head(100)

🔍 Validación de columnas:
   ✅ Producto
   ✅ Cantidad
   ✅ Unidad
   ✅ Valor_Unitario
   ✅ Total
   ✅ tienda
   ✅ fecha
   ✅ mes
   ✅ año

🔍 Valores nulos por columna:
Producto          0
Cantidad          0
Unidad            0
Total             0
Valor_Unitario    0
tienda            0
fecha             0
mes               0
año               0
dtype: int64

📋 Vista previa del dataset estandarizado:


,Producto,Cantidad,Unidad,Total,Valor_Unitario,tienda,fecha,mes,año
0,yogurt griego,1.00,und,7300.0,7300.000000,d1,2026-01-11,1,2026
1,jamón pietrán,1.00,und,13650.0,13650.000000,d1,2026-01-11,1,2026
2,queso parmesano,1.00,und,14850.0,14850.000000,d1,2026-01-11,1,2026
3,quesillo tajado,1.00,und,9990.0,9990.000000,d1,2026-01-11,1,2026
4,topping mediano,4.00,und,19800.0,4950.000000,d1,2026-01-11,1,2026
...,...,...,...,...,...,...,...,...,...
89,champiñón tajado 150g,1.00,und,5990.0,5990.000000,surtiplaza,2026-01-09,1,2026
90,papaya común,2.57,kg,8687.0,3380.155642,surtiplaza,2026-01-09,1,2026
91,yogurt griego,1.00,und,7300.0,7300.000000,d1,2026-01-21,1,2026
92,topping,3.00,und,14850.0,4950.000000,d1,2026-01-21,1,2026


## 5. Asegurar tipos de datos correctos

In [12]:
# Convertir tipos de datos
df_standardized['Producto'] = df_standardized['Producto'].astype(str)
df_standardized['Cantidad'] = pd.to_numeric(df_standardized['Cantidad'], errors='coerce')
df_standardized['Unidad'] = df_standardized['Unidad'].astype(str)
df_standardized['Valor_Unitario'] = pd.to_numeric(df_standardized['Valor_Unitario'], errors='coerce')
df_standardized['Total'] = pd.to_numeric(df_standardized['Total'], errors='coerce')
df_standardized['tienda'] = df_standardized['tienda'].astype(str)
df_standardized['fecha'] = df_standardized['fecha'].astype(str)
df_standardized['mes'] = df_standardized['mes'].astype(int)
df_standardized['año'] = df_standardized['año'].astype(int)

print("✅ Tipos de datos convertidos:")
print(df_standardized.dtypes)

✅ Tipos de datos convertidos:
Producto              str
Cantidad          float64
Unidad                str
Total             float64
Valor_Unitario    float64
tienda                str
fecha                 str
mes                 int64
año                 int64
dtype: object


## 6. Estadísticas del dataset estandarizado

In [13]:
print("📊 Estadísticas del dataset estandarizado:\n")
print(f"Total de registros: {len(df_standardized)}")
print(f"\nTiendas: {df_standardized['tienda'].nunique()}")
print(df_standardized['tienda'].value_counts())
print(f"\nFechas: {df_standardized['fecha'].nunique()}")
print(sorted(df_standardized['fecha'].unique()))
print(f"\nProductos únicos: {df_standardized['Producto'].nunique()}")
print(f"\nTotal gastado: ${df_standardized['Total'].sum():,.2f}")

📊 Estadísticas del dataset estandarizado:

Total de registros: 94

Tiendas: 3
tienda
surtiplaza     46
d1             45
mercacentro     3
Name: count, dtype: int64

Fechas: 9
['2026-01-03', '2026-01-06', '2026-01-09', '2026-01-11', '2026-01-14', '2026-01-15', '2026-01-19', '2026-01-21', '2026-01-27']

Productos únicos: 74

Total gastado: $1,037,309.00


## 7. Guardar dataset estandarizado

In [15]:
# Guardar como parquet
df_standardized.to_csv(OUTPUT_FILE, index=False)

print(f"\n💾 Archivo guardado en: {OUTPUT_FILE}")
print(f"   - Tamaño: {OUTPUT_FILE.stat().st_size / 1024:.2f} KB")
print(f"\n✅ ESTANDARIZACIÓN COMPLETADA")


💾 Archivo guardado en: ../data/staging/enero_2026_standardized.csv
   - Tamaño: 6.21 KB

✅ ESTANDARIZACIÓN COMPLETADA


## Resumen

Este notebook:
1. ✅ Cargó datos raw de staging
2. ✅ Identificó diferentes esquemas de columnas
3. ✅ Mapeó todas las columnas a un schema estándar
4. ✅ Calculó columnas faltantes (Total cuando no existía)
5. ✅ Aseguró tipos de datos correctos
6. ✅ Guardó dataset estandarizado en staging

**Siguiente paso**: Ejecutar `03_cleaning.ipynb` para limpiar y transformar los datos